# mp-spawn-workers — ex2: non-blocking mp.spawn — ProcessContext + manual join

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `mp-spawn-workers`. Running the final beacon cell reports progress against the `Distributed: mp.spawn workers` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Distributed: mp.spawn workers` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`mp-spawn-workers`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "mp-spawn-workers"
DD_SUBTOPIC = "Distributed: mp.spawn workers"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## `mp.spawn(..., join=False)` — non-blocking + manual `pc.join()`

Ex1 used `mp.spawn(fn, args=..., nprocs=N, join=True)` — blocks the launcher until all workers exit. Most training launchers do this; it's the easiest correct form.

The deepening: `join=False` returns a `ProcessContext` immediately. The launcher can do other work (start a monitoring server, run a fast smoke test, etc.) and then `pc.join()` later:

```python
pc = mp.spawn(worker, args=(world_size, port), nprocs=world_size,
              join=False)
# ... do other launcher work concurrently ...
pc.join()   # blocks until every worker exits
```

**What `pc` exposes.** `pc.pids()` (the worker PIDs), `pc.join(timeout=...)` (returns True if all done, False if timeout). The non-blocking form is how `torchrun`'s elastic launcher monitors workers — it can detect a hang and kill the group.

**Returning errors from spawn.** If a worker raises, `pc.join()` re-raises in the launcher with a `ProcessRaisedException` that wraps the original exception + traceback. The blocking `join=True` form has the same behavior; the non-blocking form just defers the re-raise to your explicit `join` call.

### Exercise 2 — non-blocking mp.spawn — ProcessContext + manual join

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Bloom level: Apply
> LO: Apply `mp.spawn(fn, args=..., nprocs=N, join=False)` to launch workers without blocking the launcher, then later call `pc.join()` on the returned `ProcessContext` to wait for completion.
> Keywords: mp.spawn, join, ProcessContext, non-blocking, pids
> ```

**KCs targeted:** `spawn-join-false-returns-context`, `process-context-join-call`

Implement `ex2_launch_nonblocking(spawn_module, worker_fn, world_size, port)`. The non-blocking spawn pattern:

1. Call `pc = spawn_module.spawn(worker_fn, args=(world_size, port), nprocs=world_size, join=False)`. Note: `join=False`.
2. Return `pc` to the caller WITHOUT joining. The caller decides when to block via `pc.join()` later.

Signature note: `spawn_module` is injected so the test can pass in a `_FakeSpawn` mock that doesn't actually fork processes (no real subprocess spawning needed on CPU). The mock returns a `_FakeProcessContext` with a `.join(timeout=None)` method and a `.pids()` method, matching torch's real `ProcessContext` API.

**Why this matters in real code.** `torchrun`'s elastic launcher uses `join=False` so it can monitor worker liveness in a separate thread, kill the group if any worker hangs past a timeout, and restart the world from the last checkpoint. The blocking `join=True` form has no escape hatch — if one worker hangs, the launcher hangs too.

Input: `spawn_module` — torch.multiprocessing or mock; `worker_fn` — callable; `world_size`, `port` — ints.
Output: the unjoined `ProcessContext`-like object.

In [ ]:
def ex2_launch_nonblocking(spawn_module, worker_fn, world_size: int, port: int):
    pc = spawn_module.spawn(worker_fn, args=(world_size, port), nprocs=world_size, join=False)
    return pc


<details><summary>Solution</summary>

```python
def ex2_launch_nonblocking(spawn_module, worker_fn, world_size: int, port: int):
    pc = spawn_module.spawn(worker_fn, args=(world_size, port), nprocs=world_size, join=False)
    return pc
```

**`join=False` returns a `ProcessContext`.** With `join=True`, `spawn` blocks until every worker exits and returns `None`. With `join=False`, it returns immediately with a context object whose `.join(timeout=...)` you call manually. Choose `False` whenever the launcher has other work to do (monitoring, hot-restart logic, graceful shutdown handling).

**`args=(world_size, port)`, not `args=(world_size, port,)` necessary?** Python tuples don't need the trailing comma when there are ≥2 elements. `args=()` for zero args; `args=(x,)` for one (the trailing comma matters there); `args=(x, y)` for two or more.

**`pc.join(timeout=...)` returns a bool.** `True` if all workers exited cleanly within the timeout, `False` on timeout. The blocking `join=True` form raises on worker failure instead — same information, different ergonomics.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()